# Notebook de Exploracion: Metodologia Box-Jenkins

Este notebook traduce a practica el contenido del reveal `S05_MODELSTDA.html`. El objetivo es que el alumno explore como identificar, estimar, diagnosticar y pronosticar con modelos ARIMA y SARIMA usando datos sinteticos y datos reales.

Al finalizar deberias poder:
- Explicar el papel de los parametros $p, d, q, P, D, Q, s$.
- Usar los tests ADF, KPSS y Ljung-Box para justificar decisiones.
- Comparar modelos con AIC y BIC bajo el principio de parsimonia.
- Reproducir el ejemplo clasico tipo Airline mostrado en la presentacion.

## Ruta de trabajo

1. Repaso conceptual de Box-Jenkins.
2. Construccion de una serie sintetica con tendencia, estacionalidad y dependencia temporal.
3. Identificacion con graficas y tests de estacionariedad.
4. Estimacion y comparacion de modelos candidatos.
5. Diagnostico de residuales y pronostico.
6. Aplicacion al conjunto real `airline_passengers.csv`, alineado con el ejemplo del reveal.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
pd.options.display.float_format = "{:.4f}".format
RNG = np.random.default_rng(42)

## Marco teorico minimo

La metodologia Box-Jenkins parte del operador de rezago $B$, definido por $B y_t = y_{t-1}$. A partir de este operador se construyen:

- **AR(p)**: la serie depende de sus ultimos $p$ valores.
- **MA(q)**: la serie depende de los ultimos $q$ errores aleatorios.
- **ARIMA(p,d,q)**: un modelo ARMA aplicado a una serie diferenciada $d$ veces.
- **SARIMA $(p,d,q) 	imes (P,D,Q)_s$**: extiende ARIMA con componentes estacionales de periodo $s$.

Interpretacion practica de parametros:
- $p$: memoria autorregresiva de corto plazo.
- $d$: numero de diferencias regulares para estabilizar la media.
- $q$: memoria de los choques o errores.
- $P, D, Q$: analogos estacionales de $p, d, q$.
- $s$: longitud del ciclo estacional, por ejemplo $12$ en datos mensuales.

## Como determinar el orden $(p,d,q)$ en ARIMA y SARIMA

Una forma practica de elegir el orden no es adivinarlo, sino seguir una secuencia de diagnostico:

1. **Elegir $d$**.
   - Si la serie tiene tendencia o cambia de nivel, probar una diferenciacion regular.
   - Usar ADF y KPSS sobre la serie original y sobre versiones diferenciadas.
   - Parar cuando la serie luzca aproximadamente estacionaria; en la practica, $d=0$ o $d=1$ suele ser suficiente.

2. **Elegir $q$ con la ACF**.
   - La ACF ayuda a detectar cuantas colas de errores quedan en la parte no estacional.
   - Un corte rapido en los primeros rezagos sugiere un MA($q$) pequeno.

3. **Elegir $p$ con la PACF**.
   - La PACF ayuda a detectar cuanta memoria autorregresiva falta.
   - Un corte rapido en los primeros rezagos sugiere un AR($p$) pequeno.

4. **Para SARIMA, elegir primero la parte estacional**.
   - Fijar $s$ segun la frecuencia de los datos: 12 en mensuales, 4 en trimestrales, 7 en diarios con patron semanal, etc.
   - Si aparecen picos en los rezagos $s, 2s, 3s$ de la ACF o PACF, probar componentes estacionales $P, D, Q$.
   - Si hay tendencia estacional, probar una diferenciacion estacional $D=1$ antes de aumentar la complejidad.

5. **Comparar candidatos**.
   - Probar pocas combinaciones cercanas alrededor de la sospecha inicial.
   - Elegir el modelo con menor AIC o BIC que deje residuales sin autocorrelacion significativa.

**Regla de trabajo recomendada**: primero estabiliza la serie con $d$ y/o $D$, luego lee ACF/PACF para proponer $p, q, P, Q$, y al final confirma con AIC, BIC y Ljung-Box.

## Relevancia de los tests

- **ADF (Augmented Dickey-Fuller)**: la hipotesis nula es que existe raiz unitaria. Si el valor-p es pequeno, hay evidencia a favor de estacionariedad.
- **KPSS**: la hipotesis nula es que la serie es estacionaria. Si el valor-p es pequeno, hay evidencia contra la estacionariedad.
- **ADF + KPSS juntos**: permiten triangulacion. Si ADF rechaza no estacionariedad y KPSS no rechaza estacionariedad, la evidencia es consistente.
- **Ljung-Box**: se aplica a residuales. Su hipotesis nula es independencia serial; si no se rechaza, el modelo ha capturado la estructura temporal principal.
- **AIC / BIC**: no prueban verdad estadistica; comparan ajuste penalizando complejidad. BIC penaliza mas fuerte y favorece modelos mas parsimoniosos.

In [ ]:
def stationarity_report(series, name, regression="c"):
    """
    Realiza tests ADF y KPSS para evaluar estacionariedad.
    
    Parámetros:
    -----------
    series : array-like
        Serie temporal a evaluar.
    name : str
        Etiqueta descriptiva de la serie (usada en la salida).
    regression : str, default='c'
        Tipo de regresión en KPSS ('c' = constante, 'ct' = constante + tendencia).
    
    Retorna:
    --------
    pd.DataFrame
        Tabla con resultados de ADF (hip. nula: raíz unitaria) y KPSS (hip. nula: estacionaria).
        Incluye estadístico y p-valor para cada test.
    """
    series = pd.Series(series).dropna()
    adf_stat, adf_pvalue, _, _, _, _ = adfuller(series, autolag="AIC")
    kpss_stat, kpss_pvalue, _, _ = kpss(series, regression=regression, nlags="auto")
    return pd.DataFrame(
        {
            "serie": [name, name],
            "test": ["ADF", "KPSS"],
            "hipotesis_nula": [
                "Existe raiz unitaria (no estacionaria)",
                "La serie es estacionaria",
            ],
            "estadistico": [adf_stat, kpss_stat],
            "p_value": [adf_pvalue, kpss_pvalue],
        }
    )


def plot_series(series, title, ylabel="valor"):
    """
    Visualiza una serie temporal con etiquetas claras.
    
    Parámetros:
    -----------
    series : pd.Series
        Serie temporal a graficar.
    title : str
        Título del gráfico.
    ylabel : str, default='valor'
        Etiqueta del eje Y.
    """
    fig, ax = plt.subplots(figsize=(12, 4))
    series.plot(ax=ax, color="tab:blue", linewidth=2)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("tiempo")
    plt.show()


def plot_correlograms(series, lags=40, title=""):
    """
    Grafica las funciones ACF (autocorrelación) y PACF (autocorrelación parcial).
    
    Parámetros:
    -----------
    series : array-like
        Serie temporal cuyas correlogramas se desean visualizar.
    lags : int, default=40
        Número de rezagos a mostrar en cada gráfico.
    title : str, default=''
        Título adicional (se añade a "ACF" y "PACF").
    
    Notas:
    ------
    Las correlogamas ayudan a identificar órdenes p (PACF) y q (ACF) de modelos ARMA.
    """
    clean = pd.Series(series).dropna()
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(clean, lags=lags, ax=axes[0])
    plot_pacf(clean, lags=lags, ax=axes[1], method="ywm")
    axes[0].set_title(f"ACF {title}")
    axes[1].set_title(f"PACF {title}")
    plt.tight_layout()
    plt.show()


def compare_models(series, candidates, trend="n"):
    """
    Ajusta varios modelos SARIMA y compara su AIC y BIC.
    
    Parámetros:
    -----------
    series : array-like o pd.Series
        Serie temporal sobre la cual ajustar los modelos.
    candidates : dict
        Diccionario con estructura {nombre_modelo: {"order": (p,d,q), "seasonal_order": (P,D,Q,s)}}
    trend : str, default='n'
        Componente de tendencia en SARIMAX ('n' = ninguno, 'c' = constante, 't' = lineal).
    
    Retorna:
    --------
    (pd.DataFrame, dict)
        - DataFrame: tabla de modelos ordenados por BIC, con AIC y BIC.
        - dict: diccionario con resultados SARIMAX ajustados para cada modelo.
    """
    fitted = {}
    rows = []
    for name, params in candidates.items():
        model = SARIMAX(
            series,
            order=params["order"],
            seasonal_order=params["seasonal_order"],
            trend=trend,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        result = model.fit(disp=False)
        fitted[name] = result
        rows.append(
            {
                "modelo": name,
                "order": params["order"],
                "seasonal_order": params["seasonal_order"],
                "AIC": result.aic,
                "BIC": result.bic,
            }
        )
    table = pd.DataFrame(rows).sort_values(["BIC", "AIC"]).reset_index(drop=True)
    return table, fitted


def ljung_box_table(residuals, lags=(12, 24)):
    """
    Realiza el test de Ljung-Box para detectar autocorrelación en residuales.
    
    Parámetros:
    -----------
    residuals : array-like
        Residuales de un modelo ajustado.
    lags : tuple, default=(12, 24)
        Rezagos en los que realizar el test.
    
    Retorna:
    --------
    pd.DataFrame
        Tabla con estadístico Ljung-Box y p-valores para cada rezago.
        Hip. nula: no hay autocorrelación. Si p-valor > 0.05, no rechazamos independencia.
    """
    return acorr_ljungbox(pd.Series(residuals).dropna(), lags=list(lags), return_df=True)

# Parte A. Datos sinteticos

Primero construiremos una serie con tres ingredientes: tendencia, estacionalidad anual y dependencia AR(1) en los errores. Esto permite observar por que un modelo puramente ARIMA puede quedarse corto cuando hay patron estacional.

In [ ]:
n_periods = 180
dates = pd.date_range("2010-01-01", periods=n_periods, freq="MS")
innovations = RNG.normal(loc=0.0, scale=2.5, size=n_periods)
ar_component = np.zeros(n_periods)
for t in range(1, n_periods):
    ar_component[t] = 0.65 * ar_component[t - 1] + innovations[t]

trend = 0.35 * np.arange(n_periods)
seasonality = 8 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
synthetic = pd.Series(120 + trend + seasonality + ar_component, index=dates, name="synthetic_demand")
synthetic.head()

In [ ]:
plot_series(synthetic, "Serie sintetica: tendencia + estacionalidad + dependencia serial", ylabel="demanda")
pd.concat(
    [
        stationarity_report(synthetic, "serie original"),
        stationarity_report(np.log(synthetic).diff(12), "log + diferencia estacional"),
        stationarity_report(np.log(synthetic).diff().diff(12), "log + diferencia regular + estacional"),
    ],
    ignore_index=True,
)

### Lectura guiada

- Si la serie original falla en ADF y/o KPSS, todavia no conviene ajustar un ARMA directo.
- La diferenciacion regular controla tendencia.
- La diferenciacion estacional controla patrones repetitivos cada $s$ periodos.
- En datos mensuales, un pico claro en lags 12, 24, 36 de la ACF/PACF sugiere estructura estacional.

In [ ]:
synthetic_stationary = np.log(synthetic).diff().diff(12).dropna()
plot_series(synthetic_stationary, "Serie sintetica transformada para identificacion", ylabel="valor transformado")
plot_correlograms(synthetic_stationary, lags=48, title="para datos sinteticos")

## Estimacion y comparacion de modelos en datos sinteticos

Probamos un ARIMA no estacional y dos variantes SARIMA. La idea no es memorizar una receta, sino ver como AIC y BIC penalizan los parametros extra cuando no aportan suficiente mejora.

In [ ]:
synthetic_candidates = {
    "ARIMA(1,1,1)": {"order": (1, 1, 1), "seasonal_order": (0, 0, 0, 0)},
    "SARIMA(1,1,1)x(1,1,0,12)": {"order": (1, 1, 1), "seasonal_order": (1, 1, 0, 12)},
    "SARIMA(1,1,1)x(1,1,1,12)": {"order": (1, 1, 1), "seasonal_order": (1, 1, 1, 12)},
}

synthetic_comparison, synthetic_fitted = compare_models(synthetic, synthetic_candidates)
synthetic_comparison

In [ ]:
best_synthetic_name = synthetic_comparison.loc[0, "modelo"]
best_synthetic = synthetic_fitted[best_synthetic_name]

print(f"Modelo elegido en sinteticos: {best_synthetic_name}")
print()
print("Coeficientes y valores-p:")
display(pd.DataFrame({"coef": best_synthetic.params, "p_value": best_synthetic.pvalues}))
print()
print("Ljung-Box sobre residuales:")
display(ljung_box_table(best_synthetic.resid, lags=(12, 24)))
best_synthetic.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
synthetic_forecast = best_synthetic.get_forecast(steps=24)
synthetic_forecast_df = synthetic_forecast.summary_frame()
future_index = pd.date_range(synthetic.index[-1] + pd.offsets.MonthBegin(), periods=24, freq="MS")
synthetic_forecast_df.index = future_index

fig, ax = plt.subplots(figsize=(12, 4))
synthetic.plot(ax=ax, label="historia", color="tab:blue")
synthetic_forecast_df["mean"].plot(ax=ax, label="pronostico", color="tab:orange")
ax.fill_between(
    synthetic_forecast_df.index,
    synthetic_forecast_df["mean_ci_lower"],
    synthetic_forecast_df["mean_ci_upper"],
    color="tab:orange",
    alpha=0.2,
)
ax.set_title("Pronostico sobre datos sinteticos")
ax.legend()
plt.show()

# Parte B. Datos reales: ejemplo Airline del reveal

En la presentacion aparece el ejemplo clasico de pasajeros aereos mensuales. Aqui reproducimos esa idea con un archivo local `airline_passengers.csv` incluido junto al notebook.

In [ ]:
data_path = Path("airline_passengers.csv")
if not data_path.exists():
    raise FileNotFoundError("No se encontro airline_passengers.csv en la carpeta del notebook.")

airline = pd.read_csv(data_path, parse_dates=["Month"], index_col="Month")
airline = airline.asfreq("MS")
airline.head()

In [ ]:
plot_series(airline["Passengers"], "Serie real: pasajeros aereos mensuales", ylabel="pasajeros")

airline["log_passengers"] = np.log(airline["Passengers"])
airline["diff_seasonal"] = airline["log_passengers"].diff(12)
airline["diff_1_12"] = airline["log_passengers"].diff().diff(12)

pd.concat(
    [
        stationarity_report(airline["Passengers"], "serie original"),
        stationarity_report(airline["diff_seasonal"], "log + diferencia estacional"),
        stationarity_report(airline["diff_1_12"], "log + diferencia regular + estacional"),
    ],
    ignore_index=True,
)

In [ ]:
# Este bloque sigue la logica de identificacion mostrada en el reveal.
check_series = airline["diff_seasonal"].dropna()
print("ADF sobre log + diferencia estacional:")
display(stationarity_report(check_series, "log + diferencia estacional").query("test == 'ADF'"))
plot_correlograms(check_series, lags=40, title="sobre pasajeros aereos")

## Comparacion del modelo del reveal contra un ARIMA no estacional

La presentacion propone el modelo SARIMA $(0,1,1) 	imes (0,1,1)_{12}$, conocido como modelo Airline. Lo comparamos con un ARIMA(1,1,1) sin estacionalidad para discutir parsimonia y capacidad explicativa.

In [ ]:
airline_candidates = {
    "ARIMA(1,1,1)": {"order": (1, 1, 1), "seasonal_order": (0, 0, 0, 0)},
    "SARIMA(0,1,1)x(0,1,1,12)": {"order": (0, 1, 1), "seasonal_order": (0, 1, 1, 12)},
    "SARIMA(1,1,1)x(0,1,1,12)": {"order": (1, 1, 1), "seasonal_order": (0, 1, 1, 12)},
}

airline_comparison, airline_fitted = compare_models(airline["log_passengers"], airline_candidates)
airline_comparison

In [ ]:
reveal_model = airline_fitted["SARIMA(0,1,1)x(0,1,1,12)"]
arima_baseline = airline_fitted["ARIMA(1,1,1)"]

comparison_focus = pd.DataFrame(
    {
        "modelo": ["ARIMA(1,1,1)", "SARIMA(0,1,1)x(0,1,1,12)"],
        "AIC": [arima_baseline.aic, reveal_model.aic],
        "BIC": [arima_baseline.bic, reveal_model.bic],
    }
)
display(comparison_focus)

print("Coeficientes del modelo Airline:")
display(pd.DataFrame({"coef": reveal_model.params, "p_value": reveal_model.pvalues}))
print()
print("Ljung-Box sobre residuales del modelo Airline:")
display(ljung_box_table(reveal_model.resid, lags=(12, 24)))

In [ ]:
reveal_model.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()

airline_forecast = reveal_model.get_forecast(steps=24)
airline_forecast_df = airline_forecast.summary_frame()
future_index = pd.date_range(airline.index[-1] + pd.offsets.MonthBegin(), periods=24, freq="MS")
airline_forecast_df.index = future_index

fig, ax = plt.subplots(figsize=(12, 4))
airline["Passengers"].plot(ax=ax, label="historia", color="tab:blue")
np.exp(airline_forecast_df["mean"]).plot(ax=ax, label="pronostico", color="tab:orange")
ax.fill_between(
    airline_forecast_df.index,
    np.exp(airline_forecast_df["mean_ci_lower"]),
    np.exp(airline_forecast_df["mean_ci_upper"]),
    color="tab:orange",
    alpha=0.2,
)
ax.set_title("Pronostico con el modelo Airline del reveal")
ax.set_ylabel("pasajeros")
ax.legend()
plt.show()

## Conclusiones y actividades para el alumno

1. Explica por que usar solo ARIMA(1,1,1) puede ser insuficiente cuando la ACF muestra picos en multiplos de 12.
2. Modifica el orden estacional del modelo sintetico y compara como cambian AIC, BIC y Ljung-Box.
3. Repite el flujo reemplazando la diferencia estacional por otra transformacion y discute si mejoras la estacionariedad.
4. Decide si prefieres un modelo con mejor AIC o con residuales mas cercanos a ruido blanco cuando los criterios entran en tension. Justifica.

La leccion central de Box-Jenkins es que un buen modelo no solo ajusta el pasado: tambien deja residuales poco estructurados y generaliza mejor al futuro.